In [ ]:
import timeit

class Dummy:
    def __init__(self, n):
        # make n attributes
        for i in range(n):
            setattr(self, f"x{i}", i)
        # mark half as dimensionless
        self.dimensionless_vars = {f"x{i}" for i in range(n // 2)}

    # original: set difference approach
    def to_units_setdiff(self, units):
        for attr in set(vars(self).keys()) - self.dimensionless_vars:
            obj = getattr(self, attr)
            try:
                obj.to_units(units)
            except Exception:
                try:
                    obj.to(units)
                except Exception:
                    pass

    # refined: membership check approach
    def to_units_membership(self, units):
        for attr, obj in vars(self).items():
            if attr in self.dimensionless_vars:
                continue
            try:
                obj.to_units(units)
            except Exception:
                try:
                    obj.to(units)
                except Exception:
                    pass


def benchmark(n_attrs, n_runs=1000):
    d = Dummy(n_attrs)
    t1 = timeit.timeit(lambda: d.to_units_setdiff("m"), number=n_runs)
    t2 = timeit.timeit(lambda: d.to_units_membership("m"), number=n_runs)
    print(f"{n_attrs:6} attrs | setdiff: {t1:.6f}s | membership: {t2:.6f}s")


if __name__ == "__main__":
    for n in [10, 100, 1000, 10000, 100000]:
        benchmark(n)


    10 attrs | setdiff: 0.004013s | membership: 0.003362s
   100 attrs | setdiff: 0.030397s | membership: 0.023767s
  1000 attrs | setdiff: 0.285265s | membership: 0.231254s
 10000 attrs | setdiff: 3.326108s | membership: 2.415358s
